# 4a. PULL transfer (DSP 2025-1, eunomia RPC)

With an agreement in hand, the Consumer asks for the data. In **PULL** mode the Provider answers
the start with a data address (endpoint and token) and the Consumer fetches the data from it. The
connectors of this notebook authenticate against the backend with the bearer token stored in the
Keystore (`setup_catalog_policies_oauth.ipynb`).

Lifecycle walked here over `/dsp/current/transfers/rpc/*` and `/api/v1/transfer-agent/*`:
`REQUESTED` -> `STARTED` (pulling data through the dataplane proxy) -> `SUSPENDED` -> `STARTED` ->
`TERMINATED`, plus two refusals: a start sent by the Consumer (role rule) and a body that is plain
JSON (shape rule).

**Runs on its own.** Prerequisites: `setup_catalog_policies_oauth.ipynb` and
`contract_negotiation.ipynb`, both guarded.

## 1. Prerequisites

Two guards make the notebook runnable on its own and cheap to rerun in the same kernel:
- the catalog notebook runs unless the kernel already holds its variant (`catalog_setup == "bearer"`);
  it onboards the participants first if needed;
- the negotiation runs unless the kernel already holds an agreement from `contract_negotiation.ipynb` on the loaded
  dataset (`agreement_source == ("rpc", target_id)`).

In [1]:
if globals().get("catalog_setup") != "bearer":
    %run ./setup_catalog_policies_oauth.ipynb
if globals().get("agreement_source") != ("rpc", target_id):
    %run ./contract_negotiation.ipynb
ensure_demo_servers()
print(f"Transferring under agreement {agreement_id}")

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

### How the transfer RPCs and Management API work

As in the negotiation, each participant asks its own agent to send the next DSP message, through
`/dsp/current/transfers/rpc/setup-{request,start,suspension,completion,termination}`. Bodies are
eunomia JSON-LD (`@context` the eunomia context, `@type` `RpcTransfer<Kind>Message`) and the
answer is an `RpcTransferResult`: `request`, the peer's `response`, and `transferAgentModel`, the
caller's own `TransferProcessView` (`id`, `state`, `stateMetadata`, `correlation` with both pids).

The Transfer Agent and Dataplane Management APIs under `/api/v1/transfer-agent/` expose:
- `/transfer-processes/{id}`: the control-plane transfer record.
- `/transfer-messages/process/{id}`: every DSP transfer message exchanged and its state transition.
- `/dataplane/dataplane-processes/transfer-process/{id}`: the dataplane process record (`ingressConfig`, `egressConfig`, `flowControl`).
- `/dataplane/dataplane-processes/{dataplaneId}/{info,logs,events}`: the proxy ingress URL, transition logs and proxied request events.

Who may send what is fixed by the protocol (DSP 9.1): only the Consumer requests, only the
Provider starts, both may suspend, complete or terminate.

In [2]:
def transfer_step(response, expected_state):
    """Prints an RpcTransferResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["transferAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Transfer request (`rpc/setup-request`, Consumer -> Provider)

The Consumer names the agreement and the format of the distribution it wants (`http+pull`). The
Provider resolves the connector instance of that distribution and opens its process in `REQUESTED`.

In [3]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pull",
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-request", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "REQUESTED")
transfer_consumer_pid = consumer_transfer["correlation"]["consumerPid"]
transfer_provider_pid = consumer_transfer["correlation"]["providerPid"]
print(f"consumerPid: {transfer_consumer_pid}\nproviderPid: {transfer_provider_pid}")

HTTP 201 POST http://127.0.0.1:1100/dsp/current/transfers/rpc/setup-request
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "agreementId": "urn:agreement:3dbda3cc-9ebc-4dec-b490-bdf6b0e33aa6",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "format": "http+pull",
    "providerA

## 3. Start (`rpc/setup-start`, Provider -> Consumer)

The Provider's `TransferStartMessage` carries the data address the Consumer pulls from: an
endpoint of the Provider's dataplane that proxies the backend described by the PULL connector.

In [4]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-start
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
    "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:481d1f22-bb99-4aa5-9725-63f3938f0073",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
      "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f",
      "agre

## 4. Pulling data through the dataplane proxy (`/api/v1/transfer-agent/dataplane/*`)

With the transfer in `STARTED`, we query the Provider's Dataplane Management API
(`/api/v1/transfer-agent/dataplane/dataplane-processes/transfer-process/{id}` and `/{dataplaneId}/info`)
to inspect the dataplane state and proxy `ingress_url`, and then fetch the dataset through the
proxy (`/dataplane/proxy/{dataplaneId}`). The proxy injects the `BEARER_TOKEN` resolved from the
Keystore (`/ecostars/api_key`) when calling the backend on `127.0.0.1:8081`.

In [5]:
provider_dp = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/transfer-process/{provider_transfer['id']}",
    headers=json_header,
))
provider_dp_id = provider_dp["id"]
dp_info = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/{provider_dp_id}/info",
    headers=json_header,
))

pulled_data = show(requests.get(dp_info["ingress_url"]))
assert pulled_data["authHeaderPresent"] is True

HTTP 200 GET http://127.0.0.1:1200/api/v1/transfer-agent/dataplane/dataplane-processes/transfer-process/urn:transfer-process:481d1f22-bb99-4aa5-9725-63f3938f0073
{
  "id": "urn:dataplane-transfer:2a8332e6-c649-4c7d-8747-12db15e772e7",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Private",
  "transferProcessId": "urn:transfer-process:481d1f22-bb99-4aa5-9725-63f3938f0073",
  "role": "Provider",
  "interactionMode": "Pull",
  "state": "Started",
  "connectorInstanceId": "urn:connector-instance:e92ae80a-8e06-48b9-8a9c-c90e73b05375",
  "ingressConfig": {
    "HttpListener": {
      "path": "/dataplane/proxy/urn:dataplane-transfer:2a8332e6-c649-4c7d-8747-12db15e772e7",
      "token": null,
      "token_type": null
    }
  },
  "egressConfig": {
    "HttpProxy": {
      "path": "http://127.0.0.1:8081",
      "token": null,
      "token_type": null
    }
  },
  "flowControl": {
    "auth": {
      "token": "{{__RUNTIME_SECRET_{/ecostars/api_key}__}}",
      "type": "BEARER_TO

## 5. Suspend (`rpc/setup-suspension`, Provider -> Consumer)

The Provider pauses access; the data address stops answering (`HTTP 403`) until the transfer
starts again.

In [6]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-suspension", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "SUSPENDED")

suspended_pull = show(requests.get(dp_info["ingress_url"]), ok=False)
assert requests.get(dp_info["ingress_url"]).status_code == 403

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-suspension
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "code": "SUSPEND",
    "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
    "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:481d1f22-bb99-4aa5-9725-63f3938f0073",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "SUSPENDED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": [
        "Suspending for demonstration purposes"
      ],
      "code": "SUSPEND"
    },
    "correlation": {
      "identif

## 6. Resume (`rpc/setup-start`, Provider -> Consumer)

A suspended transfer resumes with another start message, again sent by the Provider.

In [7]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-start
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
    "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:481d1f22-bb99-4aa5-9725-63f3938f0073",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
      "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f",
      "agre

## 7. Role check: the Consumer cannot start

Only the Provider sends a `TransferStartMessage`. The Consumer's agent refuses the RPC before
anything goes on the wire and answers a `TransferError` naming the broken rule.

In [8]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "the Consumer must not be able to start a transfer"

HTTP 400 POST http://127.0.0.1:1100/dsp/current/transfers/rpc/setup-start
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
  "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f",
  "code": "400",
  "reason": [
    "role: only a Provider sends a TransferStartMessage",
    "state: TransferStartMessage is not allowed from state STARTED"
  ]
}


## 8. Shape check: plain JSON is refused

The RPC bodies are read as JSON-LD. Without the eunomia `@context` and an `@type` the expanded
body declares no message type, and the agent answers 400.

In [9]:
payload = {
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "a body without @context and @type must be refused"

HTTP 400 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-start
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "",
  "providerPid": "",
  "code": "400",
  "reason": [
    "body: expanded body declares no message type"
  ]
}


## 9. Termination (`rpc/setup-termination`, Consumer -> Provider)

Either side may terminate. `TERMINATED` is terminal: the data address is revoked.

In [10]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferTerminationMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
    "code": "STOPPED",
    "reason": ["Demonstration completed"],
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-termination", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "TERMINATED")

HTTP 200 POST http://127.0.0.1:1100/dsp/current/transfers/rpc/setup-termination
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferTerminationMessage",
    "code": "STOPPED",
    "consumerPid": "urn:uuid:1f63e4ee-bf53-4aac-8c2e-966105ec4f6e",
    "providerPid": "urn:uuid:4222053f-b9bf-48fd-a336-2197e76d455f",
    "reason": [
      "Demonstration completed"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:d70c55d1-6843-4422-8a4a-fe42d2276213",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Private",
    "role": "consumer",
    "protocol": "dsp2025_1",
    "state": "TERMINATED",
    "stateMetadata": {
      "attribute": "BY_CONSUMER",
      "reason": [
        "Demonstration completed"
      ],
      "code": "STOPPED"
    },
    "correlation": {
      "identifiers": {},
      "consume

## 10. Final state, messages, dataplane logs and events (`/api/v1/transfer-agent/*`)

Each agent's Management API exposes:
- `GET /api/v1/transfer-agent/transfer-processes/{id}`: final state and metadata of the transfer process.
- `GET /api/v1/transfer-agent/transfer-messages/process/{id}`: every DSP message exchanged with its state transition.
- `GET /api/v1/transfer-agent/dataplane/dataplane-processes/{dataplaneId}/logs`: the dataplane state transition log.
- `GET /api/v1/transfer-agent/dataplane/dataplane-processes/{dataplaneId}/events`: diagnostic events logged when requests traversed the proxy.

In [11]:
for name, agent_url, process in [("Consumer", data_space_consumer, consumer_transfer), ("Provider", data_space_provider, provider_transfer)]:
    proc_view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-processes/{process['id']}", headers=json_header).json()
    msgs_view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-messages/process/{process['id']}", headers=json_header).json()
    transitions = [f"{m.get('stateTransitionFrom') or 'Initial'}->{m['stateTransitionTo']}" for m in msgs_view["items"]]
    print(f"{name}: state={proc_view['state']}, messages={len(msgs_view['items'])} {transitions}")

dp_logs = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/{provider_dp_id}/logs",
    headers=json_header,
))
dp_events = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/{provider_dp_id}/events",
    headers=json_header,
))

Consumer: state=TERMINATED, messages=5 ['STARTED->TERMINATED', 'SUSPENDED->STARTED', 'STARTED->SUSPENDED', 'REQUESTED->STARTED', 'INITIAL->REQUESTED']
Provider: state=TERMINATED, messages=5 ['STARTED->TERMINATED', 'SUSPENDED->STARTED', 'STARTED->SUSPENDED', 'REQUESTED->STARTED', 'INITIAL->REQUESTED']
HTTP 200 GET http://127.0.0.1:1200/api/v1/transfer-agent/dataplane/dataplane-processes/urn:dataplane-transfer:2a8332e6-c649-4c7d-8747-12db15e772e7/logs
[
  {
    "id": "urn:dataplane-process-log:52bec9d7-528a-4bc9-8d95-3c7288a848af",
    "user_id": "system",
    "user_role": "/admin",
    "visibility": "Private",
    "dataplane_process_id": "urn:dataplane-transfer:2a8332e6-c649-4c7d-8747-12db15e772e7",
    "previous_state": null,
    "new_state": "Init",
    "trigger": "Creation",
    "reason": null,
    "created_at": "2026-10-26T20:24:30.831925Z"
  },
  {
    "id": "urn:dataplane-process-log:3c88c3d8-6eb4-43c5-8032-9a86ac67551c",
    "user_id": "system",
    "user_role": "/admin",
    "vi